# 310 · Four frequencies on one output — solution

**Level:** advanced · **Topics:** Gen2, SHFSG, output-router, calibration, device-setup · **Type:** fix the bugs

Solution of [exercise.ipynb](exercise.ipynb). The four bugs:

1. `device_options="SHFSG8"` – the RTR option is missing.
2. `added_output=...` – the keyword is `added_outputs` and takes a list.
3. Different local oscillator frequencies on channels that share a synthesizer.
4. Silent: the routes are defined on the sources instead of on the target.

In [ ]:
from laboneq.simple import *

## Device setup

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html) ·
[Instrument options](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/01_instrument_options.html)

In [ ]:
setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(
    SHFSG(uid="shfsg", address="DEV12500", device_options="SHFSG8/RTR")
)
setup.add_connections(
    "shfsg",
    create_connection(to_signal="q0/drive_f0", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0_add/drive_f1", ports="SGCHANNELS/2/OUTPUT"),
    create_connection(to_signal="q0_add/drive_f2", ports="SGCHANNELS/3/OUTPUT"),
    create_connection(to_signal="q0_add/drive_f3", ports="SGCHANNELS/4/OUTPUT"),
)

## Calibration

📖 [Calibrating signals](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/00_calibrating_signals.html) ·
[Output router tutorial](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/10_advanced_topics/tutorials/08_output_router.html) ·
[`OutputRoute` reference](https://docs.zhinst.com/labone_q_user_manual/core/reference/dsl/calibration.html)

In [ ]:
calibration = Calibration()

# Fix 2 and 4: the routes belong to the TARGET signal, as a list in `added_outputs`.
# Each route names the SOURCE signal that is added to the target.
calibration["q0/drive_f0"] = SignalCalibration(
    oscillator=Oscillator(frequency=-110e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=4.0e9),
    range=10,
    added_outputs=[
        OutputRoute(source="q0_add/drive_f1", amplitude_scaling=0.2, phase_shift=0.0),
        OutputRoute(source="q0_add/drive_f2", amplitude_scaling=0.2, phase_shift=0.0),
        OutputRoute(source="q0_add/drive_f3", amplitude_scaling=0.2, phase_shift=0.0),
    ],
)

# Fix 3: same local oscillator frequency on all signals
calibration["q0_add/drive_f1"] = SignalCalibration(
    oscillator=Oscillator(frequency=-58e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=4.0e9),
    range=10,
)
calibration["q0_add/drive_f2"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=4.0e9),
    range=10,
)
calibration["q0_add/drive_f3"] = SignalCalibration(
    oscillator=Oscillator(frequency=200e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=4.0e9),
    range=10,
)

setup.set_calibration(calibration)

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## Experiment

All four tones play together, with the same envelope.

📖 [Sections and pulses](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/index.html) ·
[Pulse amplitude and phase](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/06_pulses_amplitude_and_phase.html)

In [ ]:
drive_pulse = pulse_library.const(length=1e-6)


@dsl.experiment(signals=["f0", "f1", "f2", "f3"])
def four_tones():
    with dsl.acquire_loop_rt(count=16):
        with dsl.section(name="drive"):
            dsl.play("f0", drive_pulse, amplitude=0.2)
            dsl.play("f1", drive_pulse, amplitude=1.0)
            dsl.play("f2", drive_pulse, amplitude=1.0)
            dsl.play("f3", drive_pulse, amplitude=1.0)
        with dsl.section(name="wait"):
            dsl.delay("f0", time=1e-6)


experiment = four_tones()
experiment.set_signal_map(
    {
        "f0": "q0/drive_f0",
        "f1": "q0_add/drive_f1",
        "f2": "q0_add/drive_f2",
        "f3": "q0_add/drive_f3",
    }
)

## Compile and run

📖 [Experiment](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/05_experiment/concepts/index.html)

In [ ]:
compiled = session.compile(experiment)
results = session.run(compiled)

## Check the routes

The output simulator shows each sequencer on its own, not the sum made by the router.
So we check the routes in the calibration instead: for each signal, which other signals are added to its output?

In [ ]:
# check: "target <- sources added to it"
for name in ["q0/drive_f0", "q0_add/drive_f1", "q0_add/drive_f2", "q0_add/drive_f3"]:
    routes = setup.logical_signal_by_uid(name).calibration.added_outputs
    sources = []
    if routes is not None:
        for route in routes:
            sources.append(route.source)
    print(name, "<-", sources)

**Expected result**

```
q0/drive_f0 <- ['q0_add/drive_f1', 'q0_add/drive_f2', 'q0_add/drive_f3']
q0_add/drive_f1 <- []
q0_add/drive_f2 <- []
q0_add/drive_f3 <- []
```

## Answers

1. The three errors:
   - **Missing option.** `device_options="SHFSG8"` → `"SHFSG8/RTR"`. Without it LabOne Q refuses to use the router.
   - **`added_output`** does not exist. The keyword is `added_outputs` and takes a list of `OutputRoute`.
   - **Local oscillator mismatch.** `SGCHANNELS/2` and `SGCHANNELS/3` share one synthesizer, so 4.0 GHz and 4.6 GHz cannot be set together.
     The router adds the signals digitally, before upconversion: all tones are upconverted by the synthesizer of the *target* output. Keep the same LO on all four signals, so the calibration says what really happens.
2. **Route direction (the evil one).** The customer put the route on each `q0_add` signal with `source="q0/drive_f0"`. That adds `f0` to the three unused outputs, and nothing to output 1. The route belongs to the **target** (`q0/drive_f0`) and names the **source**.
3. Output amplitude = pulse amplitude × `amplitude_scaling`. With 0.8 × 0.2 the added tones have 0.16, and `f0` has 0.2. For 0.2 each, play the added tones with amplitude 1.0 (or raise `amplitude_scaling`). Keep the sum at or below 1 to avoid clipping.
4. One sequencer can play only one hardware oscillator at a time. With all signals on one port the compiler raises `Overlapping HW oscillators`. This is why the extra tones need their own sequencers.
5. With RTR an SHFSG4 has four extra internal sequencers without a front panel output: `SGCHANNELS/4` to `SGCHANNELS/7`. They exist only to be routed. The front panel channels `SGCHANNELS/1` to `3` also work, as on the SHFSG8.

📖 [Output router tutorial](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/10_advanced_topics/tutorials/08_output_router.html) ·
[Calibration properties](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/02_calibration_properties.html)